# Soluciones — Clase 30: Despliegue básico de modelos

Soluciones completas para los ejercicios de la clase.

> **Nota para el docente:** Compartir después de que los estudiantes hayan trabajado los ejercicios.

In [ ]:
import numpy as np
import pandas as pd
import joblib
import pickle
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

np.random.seed(42)

# Datos base
n = 400
cursos = np.random.choice(['A', 'B', 'C'], size=n, p=[0.4, 0.4, 0.2])
nota_mate = np.where(cursos == 'A', np.random.normal(7.5, 1.2, n),
    np.where(cursos == 'B', np.random.normal(7.0, 1.3, n), np.random.normal(5.8, 1.5, n))).clip(1, 10)
nota_lengua = np.where(cursos == 'A', np.random.normal(7.2, 1.1, n),
    np.where(cursos == 'B', np.random.normal(6.8, 1.2, n), np.random.normal(5.5, 1.4, n))).clip(1, 10)
asistencia = np.where(cursos == 'C', np.random.uniform(0.55, 0.85, n), np.random.uniform(0.70, 0.99, n))
edades = np.random.randint(18, 35, size=n)
prob = (nota_mate * 0.4 + nota_lengua * 0.4 + asistencia * 10 * 0.2) / 10
aprobado = (np.random.uniform(0, 1, n) < prob).astype(int)

FEATURES = ['nota_matematicas', 'nota_lengua', 'asistencia', 'edad']
df = pd.DataFrame({'nota_matematicas': nota_mate.round(1), 'nota_lengua': nota_lengua.round(1),
                   'asistencia': asistencia.round(2), 'edad': edades, 'aprobado': aprobado})

X = df[FEATURES]
y = df['aprobado']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)
modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_train_sc, y_train)
print('Datos y modelo preparados')

## Solución Ejercicio 1 y 2: Guardar y comparar joblib vs pickle

In [ ]:
# Guardar con joblib
joblib.dump(modelo, 'sol_modelo_joblib.pkl')
joblib.dump(scaler, 'sol_scaler_joblib.pkl')

# Guardar con pickle
with open('sol_modelo_pickle.pkl', 'wb') as f:
    pickle.dump(modelo, f)

# Comparar tamaños
tam_joblib = os.path.getsize('sol_modelo_joblib.pkl')
tam_pickle = os.path.getsize('sol_modelo_pickle.pkl')

print('=== Comparación joblib vs pickle ===')
print(f'Tamaño con joblib: {tam_joblib / 1024:.1f} KB')
print(f'Tamaño con pickle: {tam_pickle / 1024:.1f} KB')

# Verificar que ambos funcionan igual
modelo_joblib = joblib.load('sol_modelo_joblib.pkl')
scaler_joblib = joblib.load('sol_scaler_joblib.pkl')

with open('sol_modelo_pickle.pkl', 'rb') as f:
    modelo_pickle = pickle.load(f)

acc_joblib = accuracy_score(y_test, modelo_joblib.predict(scaler_joblib.transform(X_test)))
acc_pickle = accuracy_score(y_test, modelo_pickle.predict(X_test_sc))

print(f'\nPrecisión joblib: {acc_joblib:.2%}')
print(f'Precisión pickle: {acc_pickle:.2%}')
print(f'Resultados idénticos: {acc_joblib == acc_pickle}')

print()
print('Conclusión: ambos métodos producen el mismo resultado.')
print('joblib es ligeramente mejor para arrays numpy grandes (como Random Forest).')

## Solución Ejercicio 3: Función de predicción robusta

In [ ]:
# Cargar modelos guardados
modelo_prod = joblib.load('sol_modelo_joblib.pkl')
scaler_prod = joblib.load('sol_scaler_joblib.pkl')

def predecir_estudiante(nota_matematicas, nota_lengua, asistencia, edad):
    """
    Predice si un estudiante aprobará con validación completa.
    """
    # Validación de tipos
    for nombre, valor in [('nota_matematicas', nota_matematicas),
                           ('nota_lengua', nota_lengua),
                           ('asistencia', asistencia),
                           ('edad', edad)]:
        if not isinstance(valor, (int, float)):
            raise TypeError(f'{nombre} debe ser un número, recibí: {type(valor).__name__}')
    
    # Validación de rangos
    if not (1.0 <= nota_matematicas <= 10.0):
        raise ValueError(f'nota_matematicas debe estar entre 1 y 10, recibí: {nota_matematicas}')
    if not (1.0 <= nota_lengua <= 10.0):
        raise ValueError(f'nota_lengua debe estar entre 1 y 10, recibí: {nota_lengua}')
    if not (0.0 <= asistencia <= 1.0):
        raise ValueError(f'asistencia debe estar entre 0 y 1, recibí: {asistencia}')
    if not (15 <= edad <= 80):
        raise ValueError(f'edad debe estar entre 15 y 80, recibí: {edad}')
    
    # Preparar y predecir
    X = np.array([[nota_matematicas, nota_lengua, asistencia, edad]])
    X_scaled = scaler_prod.transform(X)
    prediccion = int(modelo_prod.predict(X_scaled)[0])
    probabilidad = float(modelo_prod.predict_proba(X_scaled)[0][1])
    
    if probabilidad > 0.80 or probabilidad < 0.20:
        confianza = 'alta'
    elif probabilidad > 0.60 or probabilidad < 0.40:
        confianza = 'media'
    else:
        confianza = 'baja'
    
    return {
        'aprobado': bool(prediccion),
        'probabilidad': round(probabilidad, 3),
        'nivel_confianza': confianza,
        'mensaje': f'Probabilidad de aprobar: {probabilidad:.1%} (confianza {confianza})'
    }

# Probar casos válidos
print('=== Casos válidos ===')
casos = [
    (8.5, 7.8, 0.95, 22, 'Estudiante destacado'),
    (6.0, 6.2, 0.80, 25, 'Estudiante promedio'),
    (4.5, 4.1, 0.62, 30, 'Estudiante en riesgo'),
]
for nota_m, nota_l, asist, edad, desc in casos:
    r = predecir_estudiante(nota_m, nota_l, asist, edad)
    print(f'{desc}: {"APROBADO" if r["aprobado"] else "REPROBADO"} | {r["probabilidad"]:.1%} | {r["nivel_confianza"]}')

# Probar manejo de errores
print()
print('=== Manejo de errores ===')
casos_error = [
    (15.0, 6.8, 0.90, 23),  # nota fuera de rango
    (7.5, 6.8, 1.5, 23),    # asistencia > 1
]
for args in casos_error:
    try:
        predecir_estudiante(*args)
    except ValueError as e:
        print(f'Error de validación capturado: {e}')

## Solución Ejercicio 7: Función de validación

In [ ]:
def validar_datos(datos):
    """
    Valida datos del estudiante.
    Retorna None si todo está bien, o un mensaje de error si no.
    """
    validaciones = [
        ('nota_matematicas', 1.0, 10.0),
        ('nota_lengua', 1.0, 10.0),
        ('asistencia', 0.0, 1.0),
        ('edad', 15, 80),
    ]
    
    for campo, minimo, maximo in validaciones:
        if campo not in datos:
            return f'Campo faltante: {campo}'
        valor = datos[campo]
        if not isinstance(valor, (int, float)):
            return f'{campo} debe ser un número, recibí: {type(valor).__name__}'
        if valor < minimo or valor > maximo:
            return f'{campo} debe estar entre {minimo} y {maximo}, recibí: {valor}'
    
    return None  # Todo válido

# Casos de prueba
casos_prueba = [
    {'nota_matematicas': 7.5, 'nota_lengua': 6.8, 'asistencia': 0.9, 'edad': 23},  # válido
    {'nota_matematicas': 15.0, 'nota_lengua': 6.8, 'asistencia': 0.9, 'edad': 23},  # nota > 10
    {'nota_matematicas': 7.5, 'nota_lengua': 'siete', 'asistencia': 0.9, 'edad': 23},  # tipo incorrecto
    {'nota_matematicas': 7.5, 'asistencia': 0.9, 'edad': 23},  # campo faltante
    {'nota_matematicas': 7.5, 'nota_lengua': 6.8, 'asistencia': 1.5, 'edad': 23},  # asistencia > 1
]

for datos in casos_prueba:
    error = validar_datos(datos)
    if error:
        print(f'ERROR: {error}')
    else:
        print('OK: datos válidos')

## API Flask mejorada con validación

Esta versión incorpora la validación de datos en el endpoint `/predict`:

In [ ]:
app_mejorada = '''
from flask import Flask, request, jsonify
import joblib
import numpy as np

app = Flask(__name__)
modelo = joblib.load('sol_modelo_joblib.pkl')
scaler = joblib.load('sol_scaler_joblib.pkl')
FEATURES = ['nota_matematicas', 'nota_lengua', 'asistencia', 'edad']
RANGOS = {
    'nota_matematicas': (1.0, 10.0),
    'nota_lengua': (1.0, 10.0),
    'asistencia': (0.0, 1.0),
    'edad': (15, 80)
}

def validar(datos):
    for campo in FEATURES:
        if campo not in datos:
            return f'Campo faltante: {campo}'
        val = datos[campo]
        if not isinstance(val, (int, float)):
            return f'{campo} debe ser número'
        mn, mx = RANGOS[campo]
        if val < mn or val > mx:
            return f'{campo} debe estar entre {mn} y {mx}'
    return None

@app.route('/health')
def health():
    return jsonify({'status': 'ok', 'version': '2.0'})

@app.route('/predict', methods=['POST'])
def predict():
    try:
        datos = request.get_json()
        error = validar(datos)
        if error:
            return jsonify({'error': error}), 400
        X = np.array([[datos[f] for f in FEATURES]])
        X_sc = scaler.transform(X)
        pred = int(modelo.predict(X_sc)[0])
        prob = float(modelo.predict_proba(X_sc)[0][1])
        return jsonify({'aprobado': bool(pred), 'probabilidad': round(prob, 3)})
    except Exception as e:
        return jsonify({'error': str(e)}), 500

if __name__ == '__main__':
    app.run(debug=True, port=5001)
'''

with open('app_mejorada.py', 'w', encoding='utf-8') as f:
    f.write(app_mejorada)

print('API mejorada guardada en app_mejorada.py')
print('Para ejecutar: python app_mejorada.py')
print('Corre en el puerto 5001 para no conflictar con app.py')